Name: Muhammad Raziq Bin Sufian

Id: 24006626

## 1) Data Format & Digital Twin Justification

### Format Chosen: JSON (JavaScript Object Notation)

Justification: JSON is used because it is a lightweight, language-independent data-interchange format, natively supported by both REST HTTP (`application/json`) and MQTT payloads. Its key-value structure lets the digital twin backend easily map incoming metrics to virtual object properties.

### Data Types & Relevance:

**Numbers (Environmental Sensors)**: Simulates IoT sensors in a server rack (Temperature, Humidity, Power). Sent via HTTP REST because these updates are periodic/on-demand rather than continuous, so a simple asynchronous request-response fits.

**Coordinates (GPS Telemetry)**: Simulates automated delivery bots moving around a campus. Sent via MQTT because location changes continuously and benefits from a persistent, low-overhead streaming connection instead of opening a new HTTP connection for every update.

## 2) MQTT Protocol Justification & Routing

### Protocol Chosen: MQTT (Message Queuing Telemetry Transport)

Justification: MQTT is used for the GPS coordinate stream because it is designed for high-frequency, continuous IoT telemetry over a lightweight, persistent connection to a broker (unlike HTTP, which opens/closes a connection per request).

Protocol Route/Interface: We use the public broker `broker.hivemq.com` on port 1883, publishing/subscribing on topic `utp/digitaltwin/telemetry/coordinates`. `loop_start()` runs the network loop in a background thread so message handling doesn't block the notebook.

In [ ]:
%pip install flask requests
%pip install paho-mqtt

## 3) Generate random digital twin data (numbers & coordinates)

In [1]:
import random
import time
import json
from datetime import datetime

def generate_number_data(source_id):
    return {
        "source_id": source_id,
        "data_type": "Environmental_Numbers",
        "timestamp": datetime.now().isoformat(),
        "payload": {
            "temperature_c": round(random.uniform(22.0, 28.0), 2),
            "humidity_pct": round(random.uniform(45.0, 55.0), 2),
            "power_usage_kw": round(random.uniform(1.2, 3.8), 2)
        }
    }

def generate_coordinate_data(source_id):
    base_lat = 4.385
    base_lon = 100.979
    return {
        "source_id": source_id,
        "data_type": "GPS_Coordinates",
        "timestamp": datetime.now().isoformat(),
        "payload": {
            "latitude": round(base_lat + random.uniform(-0.005, 0.005), 6),
            "longitude": round(base_lon + random.uniform(-0.005, 0.005), 6),
            "speed_kmh": round(random.uniform(0.0, 15.0), 1)
        }
    }

# quick test
print("=== Numbers ===")
print(json.dumps(generate_number_data("Sensor_A"), indent=2))

print("\n=== Coordinates ===")
print(json.dumps(generate_coordinate_data("Bot_1"), indent=2))

=== Numbers ===
{
  "source_id": "Sensor_A",
  "data_type": "Environmental_Numbers",
  "timestamp": "2026-07-04T09:27:39.030856",
  "payload": {
    "temperature_c": 27.01,
    "humidity_pct": 45.71,
    "power_usage_kw": 3.61
  }
}

=== Coordinates ===
{
  "source_id": "Bot_1",
  "data_type": "GPS_Coordinates",
  "timestamp": "2026-07-04T09:27:39.030856",
  "payload": {
    "latitude": 4.383437,
    "longitude": 100.977811,
    "speed_kmh": 7.7
  }
}


## 4) HTTP REST endpoint — asynchronous updates (Numbers)

In [2]:
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)
http_received_storage = []

@app.route('/api/twin/update', methods=['POST'])
def receive_twin_update():
    if not request.is_json:
        return jsonify({"error": "Payload must be JSON"}), 400
    data = request.get_json()
    http_received_storage.append(data)
    print(f"[HTTP SERVER] Received update from {data.get('source_id')}: {data}")
    return jsonify({"status": "success"}), 200

def start_flask_server():
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False, threaded=True)

server_thread = threading.Thread(target=start_flask_server, daemon=True)
server_thread.start()
time.sleep(1)  # give the server a moment to start

print("HTTP server ready at http://127.0.0.1:5000/api/twin/update")

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


HTTP server ready at http://127.0.0.1:5000/api/twin/update


## 5) MQTT endpoint — streaming updates (Coordinates)

In [3]:
import paho.mqtt.client as mqtt

mqtt_received_storage = []

MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "utp/digitaltwin/telemetry/coordinates"

def on_connect(client, userdata, flags, reason_code, properties):
    print(f"[MQTT] Connected with code {reason_code}")
    client.subscribe(MQTT_TOPIC)

def on_message(client, userdata, msg):
    payload = json.loads(msg.payload.decode('utf-8'))
    mqtt_received_storage.append(payload)
    print(f"[MQTT SUBSCRIBER] Received stream update from {payload.get('source_id')}: {payload}")

mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="utp_dt_client")
mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message
mqtt_client.connect(MQTT_BROKER, MQTT_PORT)
mqtt_client.loop_start()

time.sleep(2)  # wait for the connection to establish

## 6) Parallel transmission from 2 sources per protocol + verification

In [4]:
import concurrent.futures
import requests

http_received_storage.clear()
mqtt_received_storage.clear()

# 2 sources for HTTP REST (asynchronous numeric updates)
http_payload_1 = generate_number_data("Sensor_A")
http_payload_2 = generate_number_data("Sensor_B")

# 2 sources for MQTT (streaming coordinate updates)
mqtt_payload_1 = generate_coordinate_data("Bot_1")
mqtt_payload_2 = generate_coordinate_data("Bot_2")

def send_http(payload):
    requests.post("http://127.0.0.1:5000/api/twin/update", json=payload)

def send_mqtt(payload):
    mqtt_client.publish(MQTT_TOPIC, json.dumps(payload))

print("=== Sending data in parallel ===")
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(send_http, http_payload_1),
        executor.submit(send_http, http_payload_2),
        executor.submit(send_mqtt, mqtt_payload_1),
        executor.submit(send_mqtt, mqtt_payload_2),
    ]
    for f in futures:
        f.result()

time.sleep(2)  # let delivery settle

print("\n=== Verifying received data ===")
assert http_payload_1 in http_received_storage, "HTTP Source 1 missing/corrupted!"
assert http_payload_2 in http_received_storage, "HTTP Source 2 missing/corrupted!"
print("✅ HTTP REST: both asynchronous updates received correctly.")

assert mqtt_payload_1 in mqtt_received_storage, "MQTT Source 1 missing/corrupted!"
assert mqtt_payload_2 in mqtt_received_storage, "MQTT Source 2 missing/corrupted!"
print("✅ MQTT: both streamed updates received correctly.")

print("\n🎉 All data transmitted and verified successfully.")

=== Sending data in parallel ===
[MQTT] Connected with code Success

=== Verifying received data ===


AssertionError: HTTP Source 1 missing/corrupted!